# Quality Control for the processed parquet

In [7]:
import pandas as pd
from pathlib import Path

root = next(p for p in Path.cwd().parents if (p / "config.yml").exists())

panel = pd.read_parquet(root / "data/processed_data/analysis_panel.parquet")
print(panel.shape)
print(panel.dtypes)
panel.head(20)

(17150, 57)
YEAR                               int16
GEOGRAPHY_CODE                  category
GEOGRAPHY_NAME                  category
IS8_SECTOR                      category
EMPLOYEES                        float32
BUSINESSES                       float32
gva_per_hour                     float64
weekly_pay                       float64
employment_rate                  float64
unemployment_rate                float64
gdhi_per_head                    float64
new_enterprises                  float64
deaths_of_enterprises            float64
active_enterprises               float64
high_growth_enterprises          float64
public_transport_to_employer     float64
drive_to_employer                float64
cycle_to_employer                float64
broadband_availability           float64
4g_area_coverage                 float64
ks2_attainment                   float64
gcse_by_age_19                   float64
ofsted                           float64
persistent_absences              float64
pers

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,EMPLOYEES,BUSINESSES,gva_per_hour,weekly_pay,employment_rate,unemployment_rate,...,lq_bus,emp_share,lq_emp,growth_emp,cagr_emp,growth_bus,cagr_bus,related_variety,size_large_share,size_micro_share
0,2016,E06000001,Hartlepool,Advanced Manufacturing,1470.0,25.0,29.84,521.2,69.7,4.6,...,0.925165,0.048197,1.66415,0.010204,0.001693,-0.2,-0.036508,1.332179,0.0,1.0
1,2016,E06000001,Hartlepool,Creative Industries,450.0,110.0,29.84,521.2,69.7,4.6,...,0.360178,0.014754,0.298377,0.744444,0.097176,-0.090909,-0.01576,2.200516,0.0,1.0
2,2016,E06000001,Hartlepool,Defence,0.0,0.0,29.84,521.2,69.7,4.6,...,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,0.0,<NA>,<NA>
3,2016,E06000001,Hartlepool,Digital and Technologies,1510.0,440.0,29.84,521.2,69.7,4.6,...,1.383522,0.049508,0.701919,-0.076159,-0.013116,-0.352273,-0.069823,1.072313,0.0,1.0
4,2016,E06000001,Hartlepool,Financial Services,285.0,40.0,29.84,521.2,69.7,4.6,...,0.393501,0.009344,0.157556,-0.070175,-0.012053,0.375,0.054509,1.213008,0.0,0.75
5,2016,E06000001,Hartlepool,Life Sciences,40.0,0.0,29.84,521.2,69.7,4.6,...,0.0,0.001311,0.456549,0.0,0.0,<NA>,<NA>,0.0,<NA>,<NA>
6,2016,E06000001,Hartlepool,Professional and Business Services,2630.0,870.0,29.84,521.2,69.7,4.6,...,0.992979,0.08623,0.483872,-0.019011,-0.003194,-0.264368,-0.049884,1.830641,0.0,0.965517
7,2016,E06000002,Middlesbrough,Advanced Manufacturing,620.0,25.0,29.50,481.9,68.8,5.1,...,0.646288,0.010622,0.366756,0.153226,0.024045,0.4,0.057681,1.332179,0.0,0.8
8,2016,E06000002,Middlesbrough,Creative Industries,1320.0,180.0,29.50,481.9,68.8,5.1,...,0.411722,0.022614,0.457337,0.431818,0.06165,0.055556,0.009052,2.328951,0.0,1.0
9,2016,E06000002,Middlesbrough,Defence,0.0,0.0,29.50,481.9,68.8,5.1,...,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,0.0,<NA>,<NA>


In [8]:
missing = panel.isnull().sum()
missing_pct = (missing / len(panel) * 100).round(1)
print("Missing values:")
print(pd.DataFrame({"missing": missing, "pct": missing_pct}).query("missing > 0"))

print("\nKey indicator ranges:")
for col in ["lq_emp", "lq_bus", "growth_emp", "cagr_emp", "growth_bus", "cagr_bus", "related_variety", "size_large_share", "size_micro_share"]:
    print(f"  {col}: min={panel[col].min():.3f}, max={panel[col].max():.3f}, nulls={panel[col].isnull().sum()}")

Missing values:
                              missing   pct
gva_per_hour                      196   1.1
weekly_pay                         98   0.6
employment_rate                   294   1.7
unemployment_rate                1029   6.0
gdhi_per_head                     196   1.1
new_enterprises                   196   1.1
deaths_of_enterprises             196   1.1
active_enterprises                196   1.1
high_growth_enterprises           196   1.1
public_transport_to_employer     3479  20.3
drive_to_employer                3479  20.3
cycle_to_employer                3479  20.3
broadband_availability            196   1.1
4g_area_coverage                  196   1.1
ks2_attainment                  10682  62.3
gcse_by_age_19                   2891  16.9
ofsted                          10780  62.9
persistent_absences             10927  63.7
persistent_absences_fsm         10927  63.7
persistent_absences_cla         10976  64.0
early_years_comms                2744  16.0
early_years_lite

In [9]:
import numpy as np

# --- Structural checks ---
print("=== STRUCTURAL CHECKS ===")

# duplicates
dupes = panel.duplicated(subset=["YEAR", "GEOGRAPHY_CODE", "IS8_SECTOR"]).sum()
print(f"Duplicate rows: {dupes}")

# LADs per year
lads_per_year = panel.groupby("YEAR")["GEOGRAPHY_CODE"].nunique()
print(f"\nLADs per year:\n{lads_per_year.to_string()}")

# sectors per LAD x year
sector_counts = panel.groupby(["YEAR", "GEOGRAPHY_CODE"])["IS8_SECTOR"].nunique()
print(f"\nSectors per LAD x year — min: {sector_counts.min()}, max: {sector_counts.max()}, expected: 7")
print(f"LAD x years with fewer than 7 sectors: {(sector_counts < 7).sum()}")

# --- Value sanity checks ---
print("\n=== VALUE SANITY CHECKS ===")

# high LQ
top_lq_emp = panel.nlargest(5, "lq_emp")[["YEAR", "GEOGRAPHY_NAME", "IS8_SECTOR", "lq_emp", "EMPLOYEES"]]
print(f"\nTop 5 lq_emp:\n{top_lq_emp.to_string()}")

top_lq_bus = panel.nlargest(5, "lq_bus")[["YEAR", "GEOGRAPHY_NAME", "IS8_SECTOR", "lq_bus", "BUSINESSES"]]
print(f"\nTop 5 lq_bus:\n{top_lq_bus.to_string()}")

# high growth
top_growth = panel.nlargest(5, "growth_emp")[["YEAR", "GEOGRAPHY_NAME", "IS8_SECTOR", "growth_emp", "EMPLOYEES"]]
print(f"\nTop 5 growth_emp:\n{top_growth.to_string()}")

# size_micro_share == 1
all_micro = panel[panel["size_micro_share"] == 1]
print(f"\nRows with size_micro_share == 1: {len(all_micro)}")
print(all_micro["IS8_SECTOR"].value_counts().to_string())

# --- Consistency checks ---
print("\n=== CONSISTENCY CHECKS ===")

# emp_share sums to ~1 per LAD x year (excluding Total rows already removed)
emp_share_sum = panel.groupby(["YEAR", "GEOGRAPHY_CODE"])["emp_share"].sum()
print(f"\nemp_share sum per LAD x year — min: {emp_share_sum.min():.3f}, max: {emp_share_sum.max():.3f}")
print(f"LAD x years where sum > 1.05 or < 0.95: {((emp_share_sum > 1.05) | (emp_share_sum < 0.95)).sum()}")

# lq_emp national average should be ~1 per sector x year
lq_avg = panel.groupby(["YEAR", "IS8_SECTOR"])["lq_emp"].mean()
print(f"\nlq_emp mean per sector x year — min: {lq_avg.min():.3f}, max: {lq_avg.max():.3f}")

# --- Cross-variable checks ---
print("\n=== CROSS-VARIABLE CHECKS ===")

emp_no_bus = panel[(panel["EMPLOYEES"] > 0) & (panel["BUSINESSES"] == 0)]
bus_no_emp = panel[(panel["BUSINESSES"] > 0) & (panel["EMPLOYEES"] == 0)]
print(f"Rows with EMPLOYEES > 0 but BUSINESSES == 0: {len(emp_no_bus)}")
print(f"Rows with BUSINESSES > 0 but EMPLOYEES == 0: {len(bus_no_emp)}")

if len(emp_no_bus) > 0:
    print(emp_no_bus[["YEAR", "GEOGRAPHY_NAME", "IS8_SECTOR", "EMPLOYEES", "BUSINESSES"]].head(5).to_string())
if len(bus_no_emp) > 0:
    print(bus_no_emp[["YEAR", "GEOGRAPHY_NAME", "IS8_SECTOR", "EMPLOYEES", "BUSINESSES"]].head(5).to_string())

=== STRUCTURAL CHECKS ===
Duplicate rows: 0

LADs per year:
YEAR
2016    350
2017    350
2018    350
2019    350
2020    350
2021    350
2022    350

Sectors per LAD x year — min: 7, max: 7, expected: 7
LAD x years with fewer than 7 sectors: 0

=== VALUE SANITY CHECKS ===

Top 5 lq_emp:
       YEAR  GEOGRAPHY_NAME IS8_SECTOR      lq_emp  EMPLOYEES
12490  2021  West Berkshire    Defence  120.030762     6000.0
13820  2021       Stevenage    Defence  113.804996     3000.0
5140   2018  West Berkshire    Defence  112.907858     6000.0
2690   2017  West Berkshire    Defence   111.71109     6000.0
10040  2020  West Berkshire    Defence  111.492063     6000.0

Top 5 lq_bus:
       YEAR            GEOGRAPHY_NAME IS8_SECTOR      lq_bus  BUSINESSES
3264   2017               Test Valley    Defence  108.229181         5.0
14989  2022                Portsmouth    Defence   80.014341         5.0
9872   2020  East Riding of Yorkshire    Defence   65.795634         5.0
12322  2021  East Riding of Yorks

In [10]:
# weighted mean LQ (weighted by EMPLOYEES) should be ~1
wm = panel.groupby(["YEAR", "IS8_SECTOR"]).apply(
    lambda x: np.average(x["lq_emp"].dropna(), weights=x.loc[x["lq_emp"].notna(), "EMPLOYEES"])
)
print("Employment-weighted mean lq_emp per sector x year:")
print(wm.unstack().round(3).to_string())

Employment-weighted mean lq_emp per sector x year:
IS8_SECTOR  Advanced Manufacturing  Creative Industries  Defence  Digital and Technologies  Financial Services  Life Sciences  Professional and Business Services
YEAR                                                                                                                                                             
2016                         2.318                1.821   50.710                     1.316               3.579          3.185                               1.246
2017                         2.342                1.755   54.748                     1.288               3.463          3.144                               1.240
2018                         2.404                1.754   56.177                     1.319               3.571          3.212                               1.220
2019                         2.381                1.760   59.222                     1.331               3.596          3.036              

In [11]:
# check what's in the panel before LQ is computed
# load raw employee counts and check Total row values
emp_raw = pd.read_parquet(root / "data/raw_data/employee_counts/Employee_counts_IS8_LADs.parquet")

# check geography types present
print(emp_raw["GEOGRAPHY_TYPE"].value_counts())

# check what Total rows look like after filtering to LAD only
lad_type = "local authorities: district / unitary (as of April 2023)"
emp_lad = emp_raw[emp_raw["GEOGRAPHY_TYPE"] == lad_type]
total_rows = emp_lad[emp_lad["IS8_SECTOR"] == "Total"]
print(f"\nTotal rows after LAD filter: {len(total_rows)}")
print(f"Unique years: {sorted(total_rows['YEAR'].unique())}")
print(f"\nSample national total (sum of all LAD Total rows per year):")
print(total_rows.groupby("YEAR")["OBS_VALUE"].sum())

GEOGRAPHY_TYPE
local authorities: district / unitary (as of April 2023)    308000
countries                                                     4400
Name: count, dtype: int64

Total rows after LAD filter: 3500
Unique years: [np.int16(2015), np.int16(2016), np.int16(2017), np.int16(2018), np.int16(2019), np.int16(2020), np.int16(2021), np.int16(2022), np.int16(2023), np.int16(2024)]

Sample national total (sum of all LAD Total rows per year):
YEAR
2015    28716755
2016    29205410
2017    29517910
2018    29706575
2019    30072005
2020    29477405
2021    30278980
2022    30929935
2023    31312460
2024    31479010
Name: OBS_VALUE, dtype: Int64


In [13]:
import sys
sys.path.insert(0, str(root))

from src.loader import load_config
from src.processor import DataProcessor
import pandas as pd

config = load_config(str(root / "config.yml"))
processor = DataProcessor(config)

emp = processor.loader.load_employee_counts_lad()
emp = processor.filter_lad_only(emp)
emp = processor.filter_years(emp)
emp = processor.standardise_sector_names(emp)
emp = processor.aggregate_to_is8(emp)
emp = emp.rename(columns={"OBS_VALUE": "EMPLOYEES"})

# check Total rows
total = emp[emp["IS8_SECTOR"] == "Total"]
print(f"Total rows in panel: {len(total)}")
print(f"\nNational IS8 emp (sum of all LADs per sector per year, excl Total):")
nat = emp[emp["IS8_SECTOR"] != "Total"].groupby(["YEAR", "IS8_SECTOR"])["EMPLOYEES"].sum()
print(nat.unstack().to_string())

print(f"\nNational total emp (sum of all LAD Total rows per year):")
print(total.groupby("YEAR")["EMPLOYEES"].sum())

Total rows in panel: 2450

National IS8 emp (sum of all LADs per sector per year, excl Total):
IS8_SECTOR  Advanced Manufacturing  Creative Industries  Defence  Digital and Technologies  Financial Services  Life Sciences  Professional and Business Services
YEAR                                                                                                                                                             
2016                        845840              1444145    17275                   2059935             1732105          83895                             5204620
2017                        861710              1457425    16875                   2058170             1710310          88795                             5246795
2018                        857790              1474625    16325                   2059015             1703490          90510                             5301125
2019                        862915              1528090    16550                   2158200     

In [14]:
# check if IS8 sector rows sum correctly vs what we'd expect
# Defence national total should be sum across all LADs
defence_check = emp[emp["IS8_SECTOR"] == "Defence"].groupby("YEAR")["EMPLOYEES"].sum()
print("Defence national total from LAD sum:")
print(defence_check)

# now check what Total rows look like per LAD
print("\nSample Total rows:")
print(total[total["GEOGRAPHY_CODE"] == "E06000001"].sort_values("YEAR")[["YEAR", "GEOGRAPHY_CODE", "EMPLOYEES"]])

# check if IS8 sectors sum to Total per LAD x year
is8_sum = emp[emp["IS8_SECTOR"] != "Total"].groupby(["YEAR", "GEOGRAPHY_CODE"])["EMPLOYEES"].sum().reset_index(name="IS8_SUM")
total_check = emp[emp["IS8_SECTOR"] == "Total"][["YEAR", "GEOGRAPHY_CODE", "EMPLOYEES"]].rename(columns={"EMPLOYEES": "TOTAL"})
check = is8_sum.merge(total_check, on=["YEAR", "GEOGRAPHY_CODE"])
check["diff"] = check["IS8_SUM"] - check["TOTAL"]
print(f"\nIS8 sum vs Total — mean diff: {check['diff'].mean():.1f}, max diff: {check['diff'].max():.1f}, min diff: {check['diff'].min():.1f}")

Defence national total from LAD sum:
YEAR
2016    17275
2017    16875
2018    16325
2019    16550
2020    16930
2021    16660
2022    17670
Name: EMPLOYEES, dtype: Int64

Sample Total rows:
       YEAR GEOGRAPHY_CODE  EMPLOYEES
7      2016      E06000001      30500
2807   2017      E06000001      29875
5607   2018      E06000001      28850
8407   2019      E06000001      29625
11207  2020      E06000001      28475
14007  2021      E06000001      31720
16807  2022      E06000001      29340

IS8 sum vs Total — mean diff: -51954.2, max diff: 256555.0, min diff: -311440.0


In [15]:
# what does NAT_IS8_EMP look like in the LQ computation?
# it's computed as sum of IS8 sector employees across all LADs per year
# but this excludes non-IS8 employment — correct for numerator
# let's check the actual LQ formula with real numbers for one sector/year

year = 2016
sector = "Defence"

lad_is8 = emp[(emp["YEAR"] == year) & (emp["IS8_SECTOR"] == sector)][["GEOGRAPHY_CODE", "EMPLOYEES"]]
lad_total = emp[(emp["YEAR"] == year) & (emp["IS8_SECTOR"] == "Total")][["GEOGRAPHY_CODE", "EMPLOYEES"]].rename(columns={"EMPLOYEES": "TOTAL"})
nat_is8 = lad_is8["EMPLOYEES"].sum()
nat_total = lad_total["TOTAL"].sum()

check = lad_is8.merge(lad_total, on="GEOGRAPHY_CODE")
check["lq"] = (check["EMPLOYEES"] / check["TOTAL"]) / (nat_is8 / nat_total)
print(f"NAT_IS8_EMP: {nat_is8}")
print(f"NAT_TOTAL_EMP: {nat_total}")
print(f"National share: {nat_is8/nat_total:.6f}")
print(f"\nTop 10 LQ values:")
print(check.nlargest(10, "lq")[["GEOGRAPHY_CODE", "EMPLOYEES", "TOTAL", "lq"]].to_string())

NAT_IS8_EMP: 17275
NAT_TOTAL_EMP: 29205410
National share: 0.000591

Top 10 LQ values:
    GEOGRAPHY_CODE  EMPLOYEES   TOTAL          lq
34       E06000037       6000   96850  104.736215
224      E07000243       2000   44150   76.585145
346      W06000021        600   35100   28.899437
89       E07000066       1000   84400   20.031008
19       E06000020        900   83450   18.233138
52       E06000056        810  106300   12.882407
307      S12000021        300   40800   12.431008
111      E07000088        150   20405   12.427962
24       E06000025        800  144850    9.337202
107      E07000084        400   84500    8.002921


In [16]:
# correct weighted mean — weight by LAD employment share of national total
for year in [2016, 2022]:
    for sector in panel["IS8_SECTOR"].unique():
        subset = panel[(panel["YEAR"] == year) & (panel["IS8_SECTOR"] == sector)].copy()
        nat_total = subset["EMPLOYEES"].sum()
        if nat_total == 0:
            continue
        subset["w"] = subset["EMPLOYEES"] / nat_total
        wm = (subset["lq_emp"] * subset["w"]).sum()
        print(f"{year} {sector}: {wm:.4f}")

2016 Advanced Manufacturing: 2.3183
2016 Creative Industries: 1.8210
2016 Defence: 50.7104
2016 Digital and Technologies: 1.3165
2016 Financial Services: 3.5794
2016 Life Sciences: 3.1852
2016 Professional and Business Services: 1.2459
2022 Advanced Manufacturing: 2.3058
2022 Creative Industries: 1.8838
2022 Defence: 55.9494
2022 Digital and Technologies: 1.4054
2022 Financial Services: 3.7218
2022 Life Sciences: 3.1815
2022 Professional and Business Services: 1.2733


In [17]:
# what's the true national IS8 employment from raw data?
emp_raw = processor.loader.load_employee_counts_lad()
emp_raw = processor.filter_years(emp_raw)
emp_raw = processor.standardise_sector_names(emp_raw)
emp_raw = processor.aggregate_to_is8(emp_raw)
emp_raw = emp_raw.rename(columns={"OBS_VALUE": "EMPLOYEES"})

# include ALL geographies (countries + LADs)
nat_is8_raw = emp_raw[emp_raw["IS8_SECTOR"] != "Total"].groupby(["YEAR", "IS8_SECTOR"])["EMPLOYEES"].sum()
nat_total_raw = emp_raw[emp_raw["IS8_SECTOR"] == "Total"].groupby("YEAR")["EMPLOYEES"].sum()

print("National IS8 emp from raw (all geographies):")
print(nat_is8_raw.unstack().to_string())
print("\nNational total emp from raw:")
print(nat_total_raw)

National IS8 emp from raw (all geographies):
IS8_SECTOR  Advanced Manufacturing  Creative Industries  Defence  Digital and Technologies  Financial Services  Life Sciences  Professional and Business Services
YEAR                                                                                                                                                             
2016                       3336740              5710500    66345                   8091860             6784990         328490                            20522120
2017                       3385805              5758565    64945                   8091610             6722745         348695                            20682045
2018                       3366825              5827440    61705                   8073915             6671680         355060                            20835875
2019                       3390865              6028305    61530                   8446800             6863650         378050                    

In [19]:
emp_check = processor.loader.load_employee_counts_lad()
emp_check = processor.filter_years(emp_check)
emp_check = processor.standardise_sector_names(emp_check)

print(emp_check["GEOGRAPHY_TYPE"].unique())
print("\nCountry-level geographies:")
print(emp_check[emp_check["GEOGRAPHY_TYPE"] == "countries"]["GEOGRAPHY_NAME"].unique())

# check national total from country rows only
country_total = emp_check[
    (emp_check["GEOGRAPHY_TYPE"] == "countries") & 
    (emp_check["IS8_SECTOR"] == "Total")
].groupby("YEAR")["OBS_VALUE"].sum()
print("\nNational total from country rows:")
print(country_total)

<ArrowStringArray>
['local authorities: district / unitary (as of April 2023)', 'countries']
Length: 2, dtype: str

Country-level geographies:
<ArrowStringArray>
['England', 'Great Britain', 'England and Wales', 'Scotland', 'Wales']
Length: 5, dtype: str

National total from country rows:
YEAR
2016    85160000
2017    86159000
2018    86663000
2019    87713000
2020    86049000
2021    88397000
2022    90280000
Name: OBS_VALUE, dtype: Int64


In [20]:
# check Great Britain total
gb_total = emp_check[
    (emp_check["GEOGRAPHY_NAME"] == "Great Britain") &
    (emp_check["IS8_SECTOR"] == "Total")
].groupby("YEAR")["OBS_VALUE"].sum()
print("Great Britain total employment:")
print(gb_total)

# check Great Britain IS8 sector totals
gb_is8 = emp_check[
    (emp_check["GEOGRAPHY_NAME"] == "Great Britain") &
    (emp_check["IS8_SECTOR"] != "Total")
].groupby(["YEAR", "IS8_SECTOR"])["OBS_VALUE"].sum()
print("\nGreat Britain IS8 employment:")
print(gb_is8.unstack().to_string())

Great Britain total employment:
YEAR
2016    29216000
2017    29543000
2018    29723000
2019    30071000
2020    29498000
2021    30307000
2022    30936000
Name: OBS_VALUE, dtype: Int64

Great Britain IS8 employment:
IS8_SECTOR  Advanced Manufacturing  Creative Industries  Defence  Digital and Technologies  Financial Services  Life Sciences  Professional and Business Services
YEAR                                                                                                                                                             
2016                        847750              1448500    16000                   2062350             1730250          84000                             5214000
2017                        860000              1460750    16000                   2061375             1717000          89000                             5257000
2018                        852750              1478750    15500                   2058875             1704000          91000          